In [1]:
import torch
from diffusion.approaches.matching.prob_paths import (
    GaussianCondProbPath,
    LinearAlpha,
    LinearBeta,
)
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.sampleables.mnist_sampleable import MNISTSampleable
from diffusion.architectures.backbones.res_unet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FlowTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [4]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-13 16:00:53,808 - flow-matching - INFO - Training model with size: 2.421 MiB
Epoch 0/15: 100%|██████████| 500/500 [00:38<00:00, 12.90it/s, train_loss=0.250256]
2025-10-13 16:01:32,924 - flow-matching - INFO - val loss: 0.17098721861839294
Epoch 1/15: 100%|██████████| 500/500 [00:37<00:00, 13.18it/s, train_loss=0.161084]
2025-10-13 16:02:11,102 - flow-matching - INFO - val loss: 0.15473470091819763
Epoch 2/15: 100%|██████████| 500/500 [00:37<00:00, 13.25it/s, train_loss=0.150073]
2025-10-13 16:02:49,087 - flow-matching - INFO - val loss: 0.14381751418113708
Epoch 3/15: 100%|██████████| 500/500 [01:17<00:00,  6.43it/s, train_loss=0.143801]
2025-10-13 16:04:07,114 - flow-matching - INFO - val loss: 0.14222662150859833
Epoch 4/15: 100%|██████████| 500/500 [00:38<00:00, 13.09it/s, train_loss=0.139467]
2025-10-13 16:04:45,573 - flow-matching - INFO - val loss: 0.13287262618541718
Epoch 5/15: 100%|██████████| 500/500 [00:37<00:00, 13.18it/s, train_loss=0.135736]
2025-10-13 16:05:23,76

In [5]:
torch.save(backbone.state_dict(), "./models/backbone_flow_bili.pt")